# 🌸 Iris Flower Clustering using K-Means (Unsupervised Learning)

K-Means is a centroid-based, iterative clustering algorithm that partitions unlabeled data into $k$ distinct, non-overlapping clusters. Each observation is assigned to the cluster with the nearest mean (Euclidean distance).

In this project, we explore:
- **Exploratory Data Analysis** of Iris morphological dimensions
- **Feature Scaling** for distance-based clustering
- **Determining Optimal $k$**: The Elbow Method (WCSS/Inertia) & Silhouette Analysis
- **Cluster Visualization**: Mapping 2D projections and centroids
- **Ground Truth Validation**: Cross-tabulating unsupervised clusters with botanical species labels
- **Downstream Cluster Classification**: Training KNN to predict cluster memberships


### 1. Library Imports
Importing `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`, `KMeans`, `silhouette_score`, and `KNeighborsClassifier`.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

import warnings
warnings.filterwarnings('ignore')


### 2. Loading the Iris Dataset
Read `Iris.csv` into a pandas DataFrame and display the first records to inspect columns and features.


In [ ]:
data = pd.read_csv("Iris.csv")
df = pd.DataFrame(data)
df.head()


### 3. Structural Overview: Shape & Data Types
Inspect dataset dimensions, element size, column datatypes, and data structure.


In [ ]:
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Total Elements: {df.size}")
print("\n--- Column Data Types ---")
print(df.dtypes)


### 4. Data Quality Check: Null Values & Duplicates
Verify whether the dataset contains missing values or duplicate records.


In [ ]:
print(f"Missing Values per Column:\n{df.isna().sum()}")
print(f"\nDuplicate Records: {df.duplicated().sum()}")


### 5. Ground Truth Class Distribution (`Species`)
Examine the ground truth species frequencies. In unsupervised learning, target labels are hidden during training and used purely for cluster quality validation.


In [ ]:
species_counts = df['Species'].value_counts()
print(species_counts)

plt.figure(figsize=(6, 4))
plt.bar(species_counts.index, species_counts.values, color='royalblue')
plt.title('Iris Ground Truth Species Distribution')
plt.xlabel('Species')
plt.ylabel('Count')
plt.show()


### 6. Feature Matrix Isolation
Separate the 4 morphological continuous features (`SepalLengthCm`, `SepalWidthCm`, `PetalLengthCm`, `PetalWidthCm`) from the identifier `Id` and target `Species`.


In [ ]:
cols_to_drop = [col for col in ['Id', 'Species'] if col in df.columns]
X = df.drop(columns=cols_to_drop)
y = df['Species']

print(f"Feature Matrix Shape: {X.shape}")
X.head()


### 7. Feature Scaling (`StandardScaler`)
Standardize feature vectors using `StandardScaler` so that features with larger ranges do not disproportionately dominate Euclidean distance calculations.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled.head()


### 8. Finding Optimal $k$: The Elbow Method
Compute the Within-Cluster Sum of Squares (WCSS / Inertia) across candidate cluster counts $k \in [1, 10]$:
$$WCSS = \sum_{i=1}^{k} \sum_{x \in C_i} ||x - \mu_i||^2$$
The 'elbow' point indicates diminishing returns in variance reduction as more clusters are added.


In [ ]:
wcss = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 4.5))
plt.plot(k_range, wcss, color='black', marker='o', linewidth=2)
plt.title('The Elbow Method for Optimal k (Iris Dataset)')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS (Inertia)')
plt.xticks(k_range)
plt.grid(True, linestyle='--', alpha=0.5)
plt.axvline(x=3, color='crimson', linestyle='--', label='Elbow Point (k = 3)')
plt.legend()
plt.show()


### 9. Cluster Quality Validation: Silhouette Score
The Silhouette Score measures how similar an observation is to its own cluster compared to other clusters ($s \in [-1, +1]$). Evaluate silhouette scores for $k \in [2, 8]$.


In [ ]:
sil_scores = []
k_sil_range = range(2, 9)

for k in k_sil_range:
    km = KMeans(n_clusters=k, init='k-means++', random_state=42)
    preds = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, preds)
    sil_scores.append(score)
    print(f"k = {k}: Silhouette Score = {score:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(k_sil_range, sil_scores, marker='s', color='teal', linewidth=2)
plt.title('Silhouette Scores Across Candidate Cluster Counts')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Coefficient')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


### 10. Training the Final K-Means Model ($k = 3$)
Fit the K-Means algorithm with the optimal $k=3$ clusters using `k-means++` initialization to ensure balanced, well-dispersed initial centroids.


In [ ]:
optimal_k = 3
kmeans_model = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42)
df['Cluster'] = kmeans_model.fit_predict(X_scaled)

print(f"Optimal k={optimal_k} Silhouette Score: {silhouette_score(X_scaled, df['Cluster']):.4f}")
print("\nCluster Distribution:")
print(df['Cluster'].value_counts())


### 11. Visualizing Clusters in 2D Feature Space
Plot the clusters across petal dimensions (`PetalLengthCm` vs `PetalWidthCm`) and sepal dimensions (`SepalLengthCm` vs `SepalWidthCm`), highlighting the cluster centroids.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Petal Dimensions
sns.scatterplot(
    x=df['PetalLengthCm'], y=df['PetalWidthCm'],
    hue=df['Cluster'], palette='Set1', s=70, ax=axes[0]
)
axes[0].set_title('K-Means Clusters: Petal Length vs. Petal Width')
axes[0].set_xlabel('Petal Length (cm)')
axes[0].set_ylabel('Petal Width (cm)')

# Sepal Dimensions
sns.scatterplot(
    x=df['SepalLengthCm'], y=df['SepalWidthCm'],
    hue=df['Cluster'], palette='Set1', s=70, ax=axes[1]
)
axes[1].set_title('K-Means Clusters: Sepal Length vs. Sepal Width')
axes[1].set_xlabel('Sepal Length (cm)')
axes[1].set_ylabel('Sepal Width (cm)')

plt.tight_layout()
plt.show()


### 12. Ground Truth Alignment: Contingency Matrix
Cross-tabulate the unsupervised cluster assignments against the actual botanical species labels to assess cluster purity.


In [ ]:
contingency_matrix = pd.crosstab(df['Species'], df['Cluster'], rownames=['Actual Species'], colnames=['Cluster'])

plt.figure(figsize=(6, 4.5))
sns.heatmap(contingency_matrix, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Contingency Matrix: Actual Species vs. K-Means Cluster')
plt.show()

contingency_matrix


### 13. Downstream Modeling: Predicting Cluster Labels with KNN
Train a K-Nearest Neighbors classifier to learn the decision boundaries of the discovered clusters on a train/test split.


In [ ]:
input_features = X_scaled.values
cluster_labels = df['Cluster'].values

X_train, X_test, y_train, y_test = train_test_split(
    input_features, cluster_labels, test_size=0.3, random_state=42
)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
knn_preds = knn.predict(X_test)

k_score = accuracy_score(y_test, knn_preds)
print(f"KNN Cluster Classification Accuracy: {k_score * 100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_test, knn_preds))


### 14. Key Takeaways & Unsupervised Insights
1. **Cluster Separability**: *Iris-setosa* is cleanly separated from the other two species in Euclidean space due to distinctly smaller petal dimensions.
2. **Cluster Overlap**: *Iris-versicolor* and *Iris-virginica* exhibit mild overlap along sepal dimensions, which is accurately reflected in both the Elbow curve and the contingency matrix.
3. **Optimal $k=3$**: Both the mathematical Elbow method and botanical domain knowledge converge on 3 clusters as the optimal partitioning.
